## Importing Required Libraries

In [1]:
import os
import re
import glob
import time
import subprocess
import pandas as pd

## Defining Dataset folder to work with

In [3]:
ATTACK_ROOT = "./CIC/raw_data/attack-data/"         # your attack tree root
OUTPUT_DIR  = "./CIC/AttacksCSV/"    # where per-pcap CSVs go
MERGED_CSV  = os.path.join(OUTPUT_DIR, "merged_attacks.csv")

## Columns Name

In [7]:
TSHARK_FIELDS = [
    'frame.number', 'frame.time_epoch', 'frame.len', 'eth.src', 'eth.dst', 'eth.type',
    'ip.src', 'ip.dst', 'ip.proto', 'tcp.srcport', 'tcp.dstport', 'tcp.stream',
    'tcp.window_size', 'tcp.len', 'udp.srcport', 'udp.dstport', 'udp.stream',
    'dns.qry.name', 'http', 'ntp'
]
COLUMN_NAMES = TSHARK_FIELDS[:]

#### PCAP to CSV Function

In [9]:
def parse_attack_metadata(pcap_path: str):
    """
    Expected examples:
      ./6-Attacks/1-Flood/ArloQ Camera/UDP/file.pcap
      ./6-Attacks/2-RTSP Brute Force/Nmap/HeimVision Camera/file.pcap
      ./6-Attacks/2-RTSP Brute Force/Hydra/SimCam/file.pcap
      ./6-Attacks/1-Flood/Philips Hue Bridge/TCP/file.pcap
    Returns: (attack_family, attack_tool, target_device, transport)
    """
    parts = os.path.normpath(pcap_path).split(os.sep)

    # Find index of "6-Attacks" then read following components defensively
    try:
        i = parts.index("6-Attacks")
    except ValueError:
        # fallback: try to locate via regex
        for k, p in enumerate(parts):
            if re.fullmatch(r"\d+-?Attacks", p, flags=re.I):
                i = k; break
        else:
            i = max(0, len(parts)-5)  # best-effort

    attack_family = parts[i+1] if i+1 < len(parts) else "UnknownFamily"
    # Next components can be [Tool] / [Device] / [Proto] in Flood vs BruteForce trees
    # Try to detect presence of tool by known family names:
    # For "1-Flood": structure looks like: Family / Device / Proto / file
    # For "2-RTSP Brute Force": Family / Tool / Device / file
    tool = "N/A"
    device = "UnknownDevice"
    transport = "Unknown"

    if "Flood" in attack_family:
        device = parts[i+2] if i+2 < len(parts) else device
        transport = parts[i+3] if i+3 < len(parts) else transport
    else:
        tool = parts[i+2] if i+2 < len(parts) else tool
        device = parts[i+3] if i+3 < len(parts) else device
        # transport may not exist for these—leave Unknown

    # Clean up transport (e.g., ensure one of TCP/UDP/HTTP/Unknown)
    tnorm = transport.upper()
    if tnorm not in {"TCP","UDP","HTTP"}:
        transport = "Unknown"
    else:
        transport = tnorm

    return attack_family, tool, device, transport

def normalize_device_label(name: str) -> str:
    """Normalize device label to match your benign labels as closely as possible."""
    s = re.sub(r"\s+", " ", name.strip())
    # quick harmonizations (extend as needed)
    s = s.replace("ArloQ", "Arlo Q").replace("Basestation", "Base Station").replace("Basestation", "Base Station")
    s = s.replace("Ring Basestation", "Ring Base Station")
    s = s.replace("Eufy Home Base", "Eufy HomeBase 2")
    s = s.replace("Philips Hue", "Philips Hue")
    return s

In [11]:
# --- tshark conversion ---
def pcap_to_csv_attack(pcap_file: str, out_csv: str):
    tmp_csv = out_csv + "._tmp"
    cmd = [
        "tshark", "-r", pcap_file, "-T", "fields", "-E", "separator=|",
    ]
    for f in TSHARK_FIELDS:
        cmd.extend(["-e", f])

    # Run tshark
    with open(tmp_csv, "w") as fh:
        subprocess.run(cmd, stdout=fh, check=False)

    # Read + attach metadata
    try:
        df = pd.read_csv(tmp_csv, sep="|", names=COLUMN_NAMES, dtype=str, engine="python")
    except pd.errors.EmptyDataError:
        print(f"[skip] empty: {pcap_file}")
        try: os.remove(tmp_csv)
        except: pass
        return None

    # Parse path metadata
    attack_family, attack_tool, target_device, transport = parse_attack_metadata(pcap_file)
    df["label"]         = normalize_device_label(target_device)
    df["attack_flag"]   = 1
    df["attack_family"] = attack_family
    df["attack_tool"]   = attack_tool
    df["transport"]     = transport

    # Ensure schema matches benign set (extra columns will be filled later by your feature code)
    df.to_csv(out_csv, index=False)
    try: os.remove(tmp_csv)
    except: pass
    return out_csv



#### Convert all PCAPs

In [13]:
# --- walk the attack tree and convert ---
def convert_attack_tree(attack_root: str, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    pcaps = glob.glob(os.path.join(attack_root, "**", "*.pcap"), recursive=True)
    print(f"Found {len(pcaps)} attack pcaps.")
    outputs = []
    for p in pcaps:
        rel = os.path.relpath(p, attack_root)
        # derive a safe filename
        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", rel)
        out_csv = os.path.join(out_dir, safe + ".csv")
        print(f"[pcap→csv] {rel} → {out_csv}")
        try:
            res = pcap_to_csv_attack(p, out_csv)
            if res: outputs.append(res)
        except Exception as e:
            print(f"[error] {p}: {e}")
    return outputs

#### Merge CSVs

In [15]:
def merge_attack_csvs(out_dir: str, merged_path: str):
    files = glob.glob(os.path.join(out_dir, "**", "*.csv"), recursive=True)
    if not files:
        print("No CSVs to merge.")
        return
    dfs = []
    for f in files:
        try:
            dfs.append(pd.read_csv(f))
        except Exception as e:
            print(f"[warn] failed to read {f}: {e}")
    merged = pd.concat(dfs, ignore_index=True)
    merged.to_csv(merged_path, index=False)
    print(f"Merged {len(files)} CSVs → {merged_path} (rows={len(merged):,})")

### Driver Code

In [17]:
t0 = time.time()
out_list = convert_attack_tree(ATTACK_ROOT, OUTPUT_DIR)
merge_attack_csvs(OUTPUT_DIR, MERGED_CSV)
print(f"Done in {time.time()-t0:.1f}s")

Found 96 attack pcaps.
[pcap→csv] 6-Attacks/2-RTSP Brute Force/Nmap/Luohe Camera/LuoheCamRSTP_2.pcap → ./CIC/AttacksCSV/6-Attacks_2-RTSP_Brute_Force_Nmap_Luohe_Camera_LuoheCamRSTP_2.pcap.csv
[pcap→csv] 6-Attacks/2-RTSP Brute Force/Nmap/Luohe Camera/LuoheCamRSTP_3.pcap → ./CIC/AttacksCSV/6-Attacks_2-RTSP_Brute_Force_Nmap_Luohe_Camera_LuoheCamRSTP_3.pcap.csv
[pcap→csv] 6-Attacks/2-RTSP Brute Force/Nmap/Luohe Camera/LuoheCamRSTP_1.pcap → ./CIC/AttacksCSV/6-Attacks_2-RTSP_Brute_Force_Nmap_Luohe_Camera_LuoheCamRSTP_1.pcap.csv
[pcap→csv] 6-Attacks/2-RTSP Brute Force/Nmap/Amcrest Camera/AmcrestCamRTSP_1.pcap → ./CIC/AttacksCSV/6-Attacks_2-RTSP_Brute_Force_Nmap_Amcrest_Camera_AmcrestCamRTSP_1.pcap.csv
[pcap→csv] 6-Attacks/2-RTSP Brute Force/Nmap/Amcrest Camera/AmcrestCamRTSP_3.pcap → ./CIC/AttacksCSV/6-Attacks_2-RTSP_Brute_Force_Nmap_Amcrest_Camera_AmcrestCamRTSP_3.pcap.csv
[pcap→csv] 6-Attacks/2-RTSP Brute Force/Nmap/Amcrest Camera/AmcrestCamRTSP_2.pcap → ./CIC/AttacksCSV/6-Attacks_2-RTSP_Bru

/var/folders/9h/68tfr4bn46xgf89px2064q7c0000gn/T/ipykernel_33511/1350693889.py:9: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(f))
/var/folders/9h/68tfr4bn46xgf89px2064q7c0000gn/T/ipykernel_33511/1350693889.py:9: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(f))
/var/folders/9h/68tfr4bn46xgf89px2064q7c0000gn/T/ipykernel_33511/1350693889.py:9: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(f))
/var/folders/9h/68tfr4bn46xgf89px2064q7c0000gn/T/ipykernel_33511/1350693889.py:9: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(f))
/var/folders/9h/68tfr4bn46xgf89px2064q7c0000gn/T/ipykernel_33511/1350693889.py:9: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import

Merged 96 CSVs → ./CIC/AttacksCSV/merged_attacks.csv (rows=30,330,507)
Done in 1800.2s
